In [3]:
# Cell 1 — Import dan konfigurasi utama

from pathlib import Path
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kwargs: x

warnings.filterwarnings("ignore")


# ============================================================
# INPUT CONFIG
# ============================================================

PREPROC_BASE_DIR = Path("/media/spell/Spell-lab/Lidar/D.Preprocessing") / "Preprocessed_GC25_Z008"
DBSCAN_INPUT_DIR = PREPROC_BASE_DIR / "GC25_Z080"


# ============================================================
# OUTPUT CONFIG
# ============================================================

DBSCAN_OUT_ROOT = Path("/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset")

DEV_OUT_ROOT = DBSCAN_OUT_ROOT / "Dataset Development"
TEST_OUT_ROOT = DBSCAN_OUT_ROOT / "Dataset Testing"
SUMMARY_DIR = DBSCAN_OUT_ROOT / "Summary"
LOG_DIR = DBSCAN_OUT_ROOT / "Logs"


# ============================================================
# FINAL DBSCAN PARAMETER
# ============================================================

DBSCAN_EPS = 0.20
DBSCAN_MIN_SAMPLES = 10
MIN_CLUSTER_POINTS = 20

DBSCAN_FEATURES = ["X_corr", "Y_corr", "Z_level"]

CORR_ZERO_CHECK_COLUMNS = ["X_corr", "Y_corr", "Z_corr"]
RAW_ZERO_CHECK_COLUMNS = ["X", "Y", "Z"]
FINITE_CHECK_COLUMNS = ["X_corr", "Y_corr", "Z_corr", "Z_level"]

DEGENERATE_EPS = 1e-6
FLOAT_FORMAT = "%.6f"
OVERWRITE_OUTPUT = True


# ============================================================
# DATASET CONFIG
# ============================================================

ACTIVITIES = ["Bungkuk", "Duduk", "Jongkok", "Jatuh"]

DEV_SUBJECTS = [
    "Adelia", "Afi", "Aswangga", "Bustan",
    "Dilia", "Eldivo", "Fathir", "Lina",
    "Manda", "Miftah", "Teguh", "Tsamara",
]

TEST_ROOMS = ["Controlled Room", "Uncontrolled Room"]

TEST_SUBJECTS = [
    "Kanaya", "Naila", "Nana", "Rega", "Zaira",
]

FILE_IDS = list(range(1, 10))


# ============================================================
# REQUIRED INPUT COLUMNS
# ============================================================

REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

OUTPUT_COLUMNS = REQUIRED_COLUMNS.copy()


print("===== APPLY DBSCAN CONFIG =====")
print(f"Input dir            : {DBSCAN_INPUT_DIR}")
print(f"Output root          : {DBSCAN_OUT_ROOT}")
print(f"DBSCAN eps           : {DBSCAN_EPS}")
print(f"DBSCAN min_samples   : {DBSCAN_MIN_SAMPLES}")
print(f"Min cluster points   : {MIN_CLUSTER_POINTS}")
print(f"DBSCAN features      : {DBSCAN_FEATURES}")
print(f"Overwrite output     : {OVERWRITE_OUTPUT}")

===== APPLY DBSCAN CONFIG =====
Input dir            : /media/spell/Spell-lab/Lidar/D.Preprocessing/Preprocessed_GC25_Z008/GC25_Z080
Output root          : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset
DBSCAN eps           : 0.2
DBSCAN min_samples   : 10
Min cluster points   : 20
DBSCAN features      : ['X_corr', 'Y_corr', 'Z_level']
Overwrite output     : True


In [4]:
# Cell 2 — Buat folder output dan simpan konfigurasi

for d in [DBSCAN_OUT_ROOT, DEV_OUT_ROOT, TEST_OUT_ROOT, SUMMARY_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

config = {
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "input_dir": str(DBSCAN_INPUT_DIR),
    "output_root": str(DBSCAN_OUT_ROOT),
    "dbscan_eps": DBSCAN_EPS,
    "dbscan_min_samples": DBSCAN_MIN_SAMPLES,
    "min_cluster_points": MIN_CLUSTER_POINTS,
    "dbscan_features": DBSCAN_FEATURES,
    "finite_check_columns": FINITE_CHECK_COLUMNS,
    "corr_zero_check_columns": CORR_ZERO_CHECK_COLUMNS,
    "raw_zero_check_columns": RAW_ZERO_CHECK_COLUMNS,
    "degenerate_eps": DEGENERATE_EPS,
    "output_columns": OUTPUT_COLUMNS,
    "overwrite_output": OVERWRITE_OUTPUT,
}

config_path = SUMMARY_DIR / "dbscan_apply_config.json"

with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

print(f"Config saved to: {config_path}")

Config saved to: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/dbscan_apply_config.json


In [5]:
# Cell 3 — Build manifest input-output

def build_manifest():
    records = []

    # Dataset Development
    for activity in ACTIVITIES:
        for subject in DEV_SUBJECTS:
            for file_id in FILE_IDS:
                file_name = f"{file_id}.csv"

                input_path = (
                    DBSCAN_INPUT_DIR
                    / "Dataset Development"
                    / activity
                    / subject
                    / file_name
                )

                output_path = (
                    DEV_OUT_ROOT
                    / activity
                    / subject
                    / file_name
                )

                summary_path = (
                    SUMMARY_DIR
                    / "per_file_frame_summary"
                    / "Dataset Development"
                    / activity
                    / subject
                    / f"{file_id}_frame_summary.csv"
                )

                records.append({
                    "dataset": "development",
                    "room": "",
                    "activity": activity,
                    "subject": subject,
                    "file_id": file_id,
                    "file_name": file_name,
                    "input_path": input_path,
                    "output_path": output_path,
                    "summary_path": summary_path,
                    "exists": input_path.exists(),
                })

    # Dataset Testing
    for room in TEST_ROOMS:
        for activity in ACTIVITIES:
            for subject in TEST_SUBJECTS:
                for file_id in FILE_IDS:
                    file_name = f"{file_id}.csv"

                    input_path = (
                        DBSCAN_INPUT_DIR
                        / "Dataset Testing"
                        / room
                        / activity
                        / subject
                        / file_name
                    )

                    output_path = (
                        TEST_OUT_ROOT
                        / room
                        / activity
                        / subject
                        / file_name
                    )

                    summary_path = (
                        SUMMARY_DIR
                        / "per_file_frame_summary"
                        / "Dataset Testing"
                        / room
                        / activity
                        / subject
                        / f"{file_id}_frame_summary.csv"
                    )

                    records.append({
                        "dataset": "testing",
                        "room": room,
                        "activity": activity,
                        "subject": subject,
                        "file_id": file_id,
                        "file_name": file_name,
                        "input_path": input_path,
                        "output_path": output_path,
                        "summary_path": summary_path,
                        "exists": input_path.exists(),
                    })

    return pd.DataFrame(records)


manifest_df = build_manifest()

missing_df = manifest_df[~manifest_df["exists"]].copy()
existing_df = manifest_df[manifest_df["exists"]].copy()

manifest_out = SUMMARY_DIR / "dbscan_manifest.csv"
missing_out = SUMMARY_DIR / "missing_input_files.csv"

manifest_save = manifest_df.copy()
for col in ["input_path", "output_path", "summary_path"]:
    manifest_save[col] = manifest_save[col].astype(str)

manifest_save.to_csv(manifest_out, index=False)

missing_save = missing_df.copy()
for col in ["input_path", "output_path", "summary_path"]:
    missing_save[col] = missing_save[col].astype(str)

missing_save.to_csv(missing_out, index=False)

print("===== MANIFEST SUMMARY =====")
print(f"Total expected files : {len(manifest_df)}")
print(f"Existing files       : {len(existing_df)}")
print(f"Missing files        : {len(missing_df)}")
print(f"Manifest saved       : {manifest_out}")
print(f"Missing saved        : {missing_out}")

if len(existing_df) == 0:
    raise FileNotFoundError("Tidak ada file input ditemukan. Cek struktur DBSCAN_INPUT_DIR.")

===== MANIFEST SUMMARY =====
Total expected files : 792
Existing files       : 792
Missing files        : 0
Manifest saved       : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/dbscan_manifest.csv
Missing saved        : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/missing_input_files.csv


In [6]:
# Cell 4 — Helper validasi kolom dan cleaning sebelum DBSCAN

def validate_required_columns(df, required_columns):
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        return False, missing_cols
    return True, []


def clean_points_before_dbscan(df):
    """
    Cleaning ringan sebelum DBSCAN:
    1. Drop NaN/Inf pada X_corr, Y_corr, Z_corr, Z_level
    2. Buang titik corrected all-zero pada X_corr, Y_corr, Z_corr
    3. Buang titik raw all-zero pada X, Y, Z
    """

    df_clean = df.copy()
    n_before = len(df_clean)

    # Pastikan numeric pada kolom penting
    numeric_cols = list(set(
        FINITE_CHECK_COLUMNS
        + CORR_ZERO_CHECK_COLUMNS
        + RAW_ZERO_CHECK_COLUMNS
        + DBSCAN_FEATURES
        + ["frame_id"]
    ))

    for col in numeric_cols:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # Drop NaN/Inf
    finite_mask = np.ones(len(df_clean), dtype=bool)

    for col in FINITE_CHECK_COLUMNS:
        finite_mask &= np.isfinite(df_clean[col].to_numpy())

    df_clean = df_clean.loc[finite_mask].copy()
    n_after_finite = len(df_clean)

    # Remove corrected all-zero
    corr_values = df_clean[CORR_ZERO_CHECK_COLUMNS].to_numpy(dtype=float)
    corr_all_zero = np.all(np.abs(corr_values) <= DEGENERATE_EPS, axis=1)
    df_clean = df_clean.loc[~corr_all_zero].copy()
    n_after_corr_zero = len(df_clean)

    # Remove raw all-zero
    raw_values = df_clean[RAW_ZERO_CHECK_COLUMNS].to_numpy(dtype=float)
    raw_all_zero = np.all(np.abs(raw_values) <= DEGENERATE_EPS, axis=1)
    df_clean = df_clean.loc[~raw_all_zero].copy()
    n_after_raw_zero = len(df_clean)

    cleaning_info = {
        "n_before_cleaning": n_before,
        "n_after_finite": n_after_finite,
        "n_after_corr_zero": n_after_corr_zero,
        "n_after_raw_zero": n_after_raw_zero,
        "n_removed_non_finite": n_before - n_after_finite,
        "n_removed_corr_zero": n_after_finite - n_after_corr_zero,
        "n_removed_raw_zero": n_after_corr_zero - n_after_raw_zero,
        "n_after_cleaning": n_after_raw_zero,
    }

    return df_clean, cleaning_info

In [7]:
# Cell 5 — Helper DBSCAN per frame

def apply_dbscan_to_frame(frame_df):
    """
    Output:
    - main_cluster_df: hanya point dari main cluster
    - summary: statistik audit frame
    """

    frame_id = frame_df["frame_id"].iloc[0]
    timestamp_min = frame_df["Timestamp"].min()
    timestamp_max = frame_df["Timestamp"].max()

    n_raw_frame = len(frame_df)

    clean_df, cleaning_info = clean_points_before_dbscan(frame_df)

    base_summary = {
        "frame_id": frame_id,
        "timestamp_min": timestamp_min,
        "timestamp_max": timestamp_max,
        "n_raw_frame": n_raw_frame,
        **cleaning_info,
    }

    if len(clean_df) < MIN_CLUSTER_POINTS:
        summary = {
            **base_summary,
            "status": "invalid_too_few_points_after_cleaning",
            "n_clusters_total": 0,
            "n_valid_clusters": 0,
            "n_noise": 0,
            "noise_ratio": np.nan,
            "main_cluster_label": -999,
            "main_cluster_points": 0,
            "main_cluster_ratio_clean": 0.0,
        }

        return clean_df.iloc[0:0][OUTPUT_COLUMNS].copy(), summary

    X_feat = clean_df[DBSCAN_FEATURES].to_numpy(dtype=float)

    db = DBSCAN(
        eps=DBSCAN_EPS,
        min_samples=DBSCAN_MIN_SAMPLES,
        metric="euclidean",
        n_jobs=-1,
    )

    labels = db.fit_predict(X_feat)

    clean_df = clean_df.copy()
    clean_df["_dbscan_label"] = labels

    unique_labels = sorted([lab for lab in np.unique(labels) if lab != -1])

    cluster_sizes = {
        int(lab): int(np.sum(labels == lab))
        for lab in unique_labels
    }

    valid_clusters = {
        lab: size
        for lab, size in cluster_sizes.items()
        if size >= MIN_CLUSTER_POINTS
    }

    n_noise = int(np.sum(labels == -1))
    noise_ratio = n_noise / len(clean_df) if len(clean_df) > 0 else np.nan

    if len(valid_clusters) == 0:
        summary = {
            **base_summary,
            "status": "invalid_no_valid_cluster",
            "n_clusters_total": len(unique_labels),
            "n_valid_clusters": 0,
            "n_noise": n_noise,
            "noise_ratio": noise_ratio,
            "main_cluster_label": -999,
            "main_cluster_points": 0,
            "main_cluster_ratio_clean": 0.0,
        }

        return clean_df.iloc[0:0][OUTPUT_COLUMNS].copy(), summary

    main_label = max(valid_clusters, key=valid_clusters.get)
    main_cluster_points = valid_clusters[main_label]
    main_cluster_ratio_clean = main_cluster_points / len(clean_df)

    main_cluster_df = clean_df.loc[clean_df["_dbscan_label"] == main_label].copy()

    # Output utama hanya kolom input, tanpa label DBSCAN
    main_cluster_df = main_cluster_df[OUTPUT_COLUMNS].copy()

    summary = {
        **base_summary,
        "status": "valid",
        "n_clusters_total": len(unique_labels),
        "n_valid_clusters": len(valid_clusters),
        "n_noise": n_noise,
        "noise_ratio": noise_ratio,
        "main_cluster_label": int(main_label),
        "main_cluster_points": int(main_cluster_points),
        "main_cluster_ratio_clean": float(main_cluster_ratio_clean),
    }

    return main_cluster_df, summary

In [8]:
# Cell 6 — Helper proses satu file

def process_one_file(row):
    input_path = Path(row["input_path"])
    output_path = Path(row["output_path"])
    summary_path = Path(row["summary_path"])

    file_info = {
        "dataset": row["dataset"],
        "room": row["room"],
        "activity": row["activity"],
        "subject": row["subject"],
        "file_id": row["file_id"],
        "input_path": str(input_path),
        "output_path": str(output_path),
        "summary_path": str(summary_path),
    }

    if not input_path.exists():
        return {
            **file_info,
            "file_status": "missing_input",
            "n_input_rows": 0,
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
            "n_frames_invalid": 0,
        }

    if output_path.exists() and not OVERWRITE_OUTPUT:
        return {
            **file_info,
            "file_status": "skipped_output_exists",
            "n_input_rows": np.nan,
            "n_output_rows": np.nan,
            "n_frames_total": np.nan,
            "n_frames_valid": np.nan,
            "n_frames_invalid": np.nan,
        }

    try:
        df = pd.read_csv(input_path)
    except Exception as e:
        return {
            **file_info,
            "file_status": f"read_error: {e}",
            "n_input_rows": 0,
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
            "n_frames_invalid": 0,
        }

    valid_cols, missing_cols = validate_required_columns(df, REQUIRED_COLUMNS)

    if not valid_cols:
        return {
            **file_info,
            "file_status": f"missing_columns: {missing_cols}",
            "n_input_rows": len(df),
            "n_output_rows": 0,
            "n_frames_total": 0,
            "n_frames_valid": 0,
            "n_frames_invalid": 0,
        }

    df = df[REQUIRED_COLUMNS].copy()
    df["frame_id"] = pd.to_numeric(df["frame_id"], errors="coerce")
    df = df.dropna(subset=["frame_id"]).copy()
    df["frame_id"] = df["frame_id"].astype(int)

    output_frames = []
    frame_summaries = []

    for frame_id, frame_df in df.groupby("frame_id", sort=True):
        main_cluster_df, summary = apply_dbscan_to_frame(frame_df)

        summary.update({
            "dataset": row["dataset"],
            "room": row["room"],
            "activity": row["activity"],
            "subject": row["subject"],
            "file_id": row["file_id"],
            "file_name": row["file_name"],
        })

        frame_summaries.append(summary)

        if len(main_cluster_df) > 0:
            output_frames.append(main_cluster_df)

    if output_frames:
        output_df = pd.concat(output_frames, ignore_index=True)
    else:
        output_df = pd.DataFrame(columns=OUTPUT_COLUMNS)

    frame_summary_df = pd.DataFrame(frame_summaries)

    output_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.parent.mkdir(parents=True, exist_ok=True)

    output_df.to_csv(output_path, index=False, float_format=FLOAT_FORMAT)
    frame_summary_df.to_csv(summary_path, index=False, float_format=FLOAT_FORMAT)

    n_frames_total = len(frame_summary_df)
    n_frames_valid = int((frame_summary_df["status"] == "valid").sum()) if n_frames_total > 0 else 0
    n_frames_invalid = n_frames_total - n_frames_valid

    return {
        **file_info,
        "file_status": "processed",
        "n_input_rows": len(df),
        "n_output_rows": len(output_df),
        "n_frames_total": n_frames_total,
        "n_frames_valid": n_frames_valid,
        "n_frames_invalid": n_frames_invalid,
        "valid_frame_ratio": n_frames_valid / n_frames_total if n_frames_total > 0 else np.nan,
        "mean_main_cluster_points": (
            frame_summary_df.loc[frame_summary_df["status"] == "valid", "main_cluster_points"].mean()
            if n_frames_valid > 0 else np.nan
        ),
        "median_main_cluster_points": (
            frame_summary_df.loc[frame_summary_df["status"] == "valid", "main_cluster_points"].median()
            if n_frames_valid > 0 else np.nan
        ),
        "mean_noise_ratio": (
            frame_summary_df["noise_ratio"].mean()
            if "noise_ratio" in frame_summary_df.columns else np.nan
        ),
    }

In [9]:
# Cell 7 — Mini test satu file dulu

# Ambil satu file existing untuk sanity check
test_row = existing_df.iloc[0].to_dict()

print("===== MINI TEST FILE =====")
print(f"Input : {test_row['input_path']}")
print(f"Output: {test_row['output_path']}")

mini_result = process_one_file(test_row)

print("\n===== MINI TEST RESULT =====")
for k, v in mini_result.items():
    print(f"{k}: {v}")

mini_output_path = Path(mini_result["output_path"])
mini_summary_path = Path(mini_result["summary_path"])

if mini_output_path.exists():
    mini_df = pd.read_csv(mini_output_path)
    print("\nMini output preview:")
    display(mini_df.head())
    print(f"Mini output shape: {mini_df.shape}")

if mini_summary_path.exists():
    mini_summary_df = pd.read_csv(mini_summary_path)
    print("\nMini frame summary preview:")
    display(mini_summary_df.head())
    print(f"Mini summary shape: {mini_summary_df.shape}")

===== MINI TEST FILE =====
Input : /media/spell/Spell-lab/Lidar/D.Preprocessing/Preprocessed_GC25_Z008/GC25_Z080/Dataset Development/Bungkuk/Adelia/1.csv
Output: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Dataset Development/Bungkuk/Adelia/1.csv

===== MINI TEST RESULT =====
dataset: development
room: 
activity: Bungkuk
subject: Adelia
file_id: 1
input_path: /media/spell/Spell-lab/Lidar/D.Preprocessing/Preprocessed_GC25_Z008/GC25_Z080/Dataset Development/Bungkuk/Adelia/1.csv
output_path: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Dataset Development/Bungkuk/Adelia/1.csv
summary_path: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/per_file_frame_summary/Dataset Development/Bungkuk/Adelia/1_frame_summary.csv
file_status: processed
n_input_rows: 65081
n_output_rows: 46065
n_frames_total: 55
n_frames_valid: 55
n_frames_invalid: 0
valid_frame_ratio: 1.0
mean_main_cluster_points: 837.5454545454545
median_main_cluster_points: 779.0
mean_noise_ratio: 6.135021470556965e-05

Mini ou

,frame_id,Timestamp,X,Y,Z,X_corr,Y_corr,Z_corr,Z_level,Reflectivity
0,0,4619236332940,0.731,0.540,0.675,0.947778,0.540,0.302824,1.189495,14.0
1,0,4619236332940,0.746,0.547,0.642,0.947427,0.547,0.266576,1.153248,11.0
2,0,4619236332940,0.765,0.563,0.617,0.954081,0.563,0.235889,1.122560,12.0
3,0,4619236332940,0.778,0.572,0.586,0.952762,0.572,0.202299,1.088971,14.0
4,0,4619236332940,0.728,0.562,0.683,0.948440,0.562,0.311342,1.198014,13.0


Mini output shape: (46065, 10)

Mini frame summary preview:


,frame_id,timestamp_min,timestamp_max,n_raw_frame,n_before_cleaning,n_after_finite,n_after_corr_zero,n_after_raw_zero,n_removed_non_finite,n_removed_corr_zero,...,noise_ratio,main_cluster_label,main_cluster_points,main_cluster_ratio_clean,dataset,room,activity,subject,file_id,file_name
0,0,4619236332940,4619335732940,1477,1477,1477,1056,1056,0,421,...,0.000000,0,1056,1.000000,development,NaN,Bungkuk,Adelia,1,1.csv
1,1,4619336212940,4619435572940,1448,1448,1448,955,955,0,493,...,0.000000,0,955,1.000000,development,NaN,Bungkuk,Adelia,1,1.csv
2,2,4619440372940,4619534932940,1457,1457,1457,1025,1025,0,432,...,0.000976,0,1024,0.999024,development,NaN,Bungkuk,Adelia,1,1.csv
3,3,4619539732940,4619634292940,1471,1471,1471,998,998,0,473,...,0.000000,0,998,1.000000,development,NaN,Bungkuk,Adelia,1,1.csv
4,4,4619639092940,4619733672940,1466,1466,1466,1033,1033,0,433,...,0.000000,0,1033,1.000000,development,NaN,Bungkuk,Adelia,1,1.csv


Mini summary shape: (55, 26)


In [10]:
# Cell 8 — Apply DBSCAN ke seluruh dataset

file_results = []

for _, row in tqdm(existing_df.iterrows(), total=len(existing_df), desc="Applying DBSCAN"):
    result = process_one_file(row)
    file_results.append(result)

file_summary_df = pd.DataFrame(file_results)

file_summary_path = SUMMARY_DIR / "dbscan_file_summary.csv"
file_summary_df.to_csv(file_summary_path, index=False, float_format=FLOAT_FORMAT)

print("===== FULL APPLY DONE =====")
print(f"Total processed records : {len(file_summary_df)}")
print(f"File summary saved      : {file_summary_path}")

display(file_summary_df["file_status"].value_counts(dropna=False).to_frame("count"))

Applying DBSCAN:   0%|          | 0/792 [00:00<?, ?it/s]

===== FULL APPLY DONE =====
Total processed records : 792
File summary saved      : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/dbscan_file_summary.csv


,count
file_status,
processed,792


In [11]:
# Cell 9 — Gabungkan semua frame summary menjadi global summary

frame_summary_paths = list((SUMMARY_DIR / "per_file_frame_summary").rglob("*_frame_summary.csv"))

all_frame_summaries = []

for path in tqdm(frame_summary_paths, desc="Merging frame summaries"):
    try:
        temp = pd.read_csv(path)
        temp["summary_file"] = str(path)
        all_frame_summaries.append(temp)
    except Exception as e:
        print(f"Failed reading {path}: {e}")

if all_frame_summaries:
    global_frame_summary_df = pd.concat(all_frame_summaries, ignore_index=True)
else:
    global_frame_summary_df = pd.DataFrame()

global_frame_summary_path = SUMMARY_DIR / "dbscan_frame_summary_global.csv"
global_frame_summary_df.to_csv(global_frame_summary_path, index=False, float_format=FLOAT_FORMAT)

print("===== GLOBAL FRAME SUMMARY =====")
print(f"Total frame summaries : {len(global_frame_summary_df)}")
print(f"Saved to              : {global_frame_summary_path}")

if len(global_frame_summary_df) > 0:
    display(global_frame_summary_df.head())
    display(global_frame_summary_df["status"].value_counts(dropna=False).to_frame("count"))

Merging frame summaries:   0%|          | 0/792 [00:00<?, ?it/s]

===== GLOBAL FRAME SUMMARY =====
Total frame summaries : 47371
Saved to              : /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/dbscan_frame_summary_global.csv


,frame_id,timestamp_min,timestamp_max,n_raw_frame,n_before_cleaning,n_after_finite,n_after_corr_zero,n_after_raw_zero,n_removed_non_finite,n_removed_corr_zero,...,main_cluster_label,main_cluster_points,main_cluster_ratio_clean,dataset,room,activity,subject,file_id,file_name,summary_file
0,0,6359557692940,6359640732940,629,629,629,449,449,0,180,...,0,426,0.948775,testing,Uncontrolled Room,Jongkok,Kanaya,4,4.csv,/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/...
1,1,6359657532940,6359745372940,627,627,627,451,451,0,176,...,0,436,0.966741,testing,Uncontrolled Room,Jongkok,Kanaya,4,4.csv,/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/...
2,2,6359762652940,6359844732940,616,616,616,452,452,0,164,...,0,428,0.946903,testing,Uncontrolled Room,Jongkok,Kanaya,4,4.csv,/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/...
3,3,6359862012940,6359949852940,602,602,602,440,440,0,162,...,0,428,0.972727,testing,Uncontrolled Room,Jongkok,Kanaya,4,4.csv,/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/...
4,4,6359967132940,6360049212940,633,633,633,458,458,0,175,...,0,430,0.938865,testing,Uncontrolled Room,Jongkok,Kanaya,4,4.csv,/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/...


,count
status,
valid,47371


In [12]:
# Cell 10 — Statistik awal point count hasil DBSCAN

if len(global_frame_summary_df) == 0:
    raise ValueError("Global frame summary kosong. Jalankan Cell 9 terlebih dahulu.")

valid_frames_df = global_frame_summary_df[
    global_frame_summary_df["status"] == "valid"
].copy()

point_counts = valid_frames_df["main_cluster_points"].dropna().astype(float)

stats = {
    "n_valid_frames": len(point_counts),
    "min": point_counts.min(),
    "max": point_counts.max(),
    "mean": point_counts.mean(),
    "std": point_counts.std(),
    "p01": point_counts.quantile(0.01),
    "p05": point_counts.quantile(0.05),
    "p10": point_counts.quantile(0.10),
    "p25": point_counts.quantile(0.25),
    "p50_median": point_counts.quantile(0.50),
    "p75": point_counts.quantile(0.75),
    "p90": point_counts.quantile(0.90),
    "p95": point_counts.quantile(0.95),
    "p99": point_counts.quantile(0.99),
}

point_count_stats_df = pd.DataFrame([stats])

point_count_stats_path = SUMMARY_DIR / "dbscan_point_count_stats.csv"
point_count_stats_df.to_csv(point_count_stats_path, index=False, float_format=FLOAT_FORMAT)

print("===== POINT COUNT STATS AFTER DBSCAN =====")
display(point_count_stats_df)
print(f"Saved to: {point_count_stats_path}")

===== POINT COUNT STATS AFTER DBSCAN =====


,n_valid_frames,min,max,mean,std,p01,p05,p10,p25,p50_median,p75,p90,p95,p99
0,47371,40.0,1666.0,433.819869,267.372343,101.0,141.0,160.0,230.0,369.0,559.0,804.0,1002.0,1282.0


Saved to: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/dbscan_point_count_stats.csv


In [13]:
# Cell 11 — Statistik per dataset, room, activity, subject

group_cols = ["dataset", "room", "activity", "subject"]

group_stats_df = (
    valid_frames_df
    .groupby(group_cols, dropna=False)["main_cluster_points"]
    .agg(
        n_valid_frames="count",
        min_points="min",
        mean_points="mean",
        median_points="median",
        std_points="std",
        max_points="max",
    )
    .reset_index()
)

group_stats_path = SUMMARY_DIR / "dbscan_point_count_stats_by_group.csv"
group_stats_df.to_csv(group_stats_path, index=False, float_format=FLOAT_FORMAT)

print("===== GROUP POINT COUNT STATS =====")
display(group_stats_df.head(20))
print(f"Saved to: {group_stats_path}")



===== GROUP POINT COUNT STATS =====


,dataset,room,activity,subject,n_valid_frames,min_points,mean_points,median_points,std_points,max_points
0,development,NaN,Bungkuk,Adelia,518,91,496.536680,482.0,277.472548,1385
1,development,NaN,Bungkuk,Afi,523,66,448.330784,416.0,252.443043,1155
2,development,NaN,Bungkuk,Aswangga,517,56,387.735010,319.0,256.743065,1269
3,development,NaN,Bungkuk,Bustan,498,79,592.461847,463.5,394.513856,1666
4,development,NaN,Bungkuk,Dilia,596,84,567.929530,488.5,316.523004,1385
5,development,NaN,Bungkuk,Eldivo,553,63,427.929476,376.0,280.885583,1466
6,development,NaN,Bungkuk,Fathir,510,71,425.029412,391.0,237.524738,1197
7,development,NaN,Bungkuk,Lina,556,101,489.645683,426.0,286.258999,1322
8,development,NaN,Bungkuk,Manda,563,40,340.797513,265.0,230.589138,1097
9,development,NaN,Bungkuk,Miftah,521,77,395.719770,347.0,248.693614,1304


Saved to: /media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Summary/dbscan_point_count_stats_by_group.csv


In [14]:
# Cell 12 — Final sanity check output

print("===== FINAL SANITY CHECK =====")

print("\nOutput root:")
print(DBSCAN_OUT_ROOT)

print("\nDevelopment output exists:", DEV_OUT_ROOT.exists())
print("Testing output exists    :", TEST_OUT_ROOT.exists())
print("Summary exists           :", SUMMARY_DIR.exists())

processed_count = (file_summary_df["file_status"] == "processed").sum()
skipped_count = (file_summary_df["file_status"] == "skipped_output_exists").sum()
error_count = len(file_summary_df) - processed_count - skipped_count

print("\nFile status:")
print(f"Processed : {processed_count}")
print(f"Skipped   : {skipped_count}")
print(f"Other     : {error_count}")

print("\nFrame status:")
display(global_frame_summary_df["status"].value_counts(dropna=False).to_frame("count"))

print("\nPoint count global stats:")
display(point_count_stats_df)

===== FINAL SANITY CHECK =====

Output root:
/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset

Development output exists: True
Testing output exists    : True
Summary exists           : True

File status:
Processed : 792
Skipped   : 0
Other     : 0

Frame status:


,count
status,
valid,47371



Point count global stats:


,n_valid_frames,min,max,mean,std,p01,p05,p10,p25,p50_median,p75,p90,p95,p99
0,47371,40.0,1666.0,433.819869,267.372343,101.0,141.0,160.0,230.0,369.0,559.0,804.0,1002.0,1282.0
